# Computer Vision Exploratory Data Analysis & Visual Analytics

[Back to Session Overview](../README.md) · [Back to Module 01](../../README.md)

Exploratory Data Analysis (EDA) in Computer Vision differs substantially from traditional tabular analytics. Instead of inspecting scalar distributions alone, computer vision practitioners must inspect image spatial resolutions, bounding box scale spectrums, spatial densities, aspect ratios, and class frequencies to ensure model architectures (such as anchor boxes in object detectors or temporal sampling rates in trackers) align with empirical data distributions.

### Learning Objectives
1. Profile image spatial geometry (heterogeneous resolutions, aspect ratios) and design spatial normalization pipelines.
2. Quantify bounding box area distributions across COCO scale tiers ($< 32^2$, $[32^2, 96^2]$, $> 96^2$).
3. Analyze bounding box aspect ratios ($W/H$) to motivate anchor box design.
4. Evaluate scene density (sparse vs. crowded detection).
5. Diagnose long-tail class imbalance and evaluate mitigation strategies.
6. Analyze trajectory continuity, scale variance, and temporal downsampling in Visual Object Tracking (VOT).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

# Ensure data path resolves relative to this notebook
DATA_DIR = Path("..") / "data"
BBOX_PATH = DATA_DIR / "bounding_boxes.csv"
TRACK_PATH = DATA_DIR / "tracking_sequences.csv"

print("Data paths configured.")
print(f"Bounding boxes path: {BBOX_PATH} (exists: {BBOX_PATH.exists()})")
print(f"Tracking sequences path: {TRACK_PATH} (exists: {TRACK_PATH.exists()})")

## 1. Loading and Inspecting Bounding Box Annotations

We load the bounding box dataset containing image identifiers, spatial dimensions, bounding box coordinates $(x_{min}, y_{min}, w, h)$, area, and scale categories.


In [ ]:
bbox_df = pl.read_csv(BBOX_PATH)
print("Bounding Box Dataset Schema:")
print(bbox_df.schema)
print(f"Total annotations loaded: {bbox_df.height}")
print(bbox_df.head(5))

## 2. Image Spatial Resolutions and Aspect Ratios

In raw datasets, source images frequently exhibit heterogeneous resolutions (e.g., 1080p, 720p, VGA). Feeding heterogeneous images into fixed-size neural network backbones (e.g., $640 \times 640$) requires understanding whether letterboxing (aspect-ratio preserving padding) or direct stretching is appropriate.


In [ ]:
# Compute unique image dimensions and their frequencies
img_resolutions = (
    bbox_df.select(["image_id", "image_width", "image_height"])
    .unique()
    .with_columns((pl.col("image_width") / pl.col("image_height")).round(3).alias("aspect_ratio"))
)

print(f"Total unique images: {img_resolutions.height}")
res_counts = img_resolutions.group_by(["image_width", "image_height", "aspect_ratio"]).len()
print("Image resolution distributions:")
print(res_counts)

## 3. Object Density: Detecting Sparse vs. Crowded Scenes

Object density is the number of bounding boxes per image. High object density signals "crowded scenes" where standard Non-Maximum Suppression (NMS) may suppress valid overlapping targets, requiring Soft-NMS or transformer-based bipartite matching.


In [ ]:
density_df = (
    bbox_df.group_by("image_id")
    .len()
    .rename({"len": "box_count"})
    .sort("box_count", descending=True)
)

print("Object density summary statistics:")
print(density_df.select("box_count").describe())

mean_density = density_df["box_count"].mean()
max_density = density_df["box_count"].max()
min_density = density_df["box_count"].min()

print(f"Mean density: {mean_density:.2f} boxes/image | Min: {min_density} | Max: {max_density}")

## 4. Multi-Scale Object Analysis: COCO Benchmark Tiers

The MS-COCO benchmark formalizes three canonical object scale categories based on pixel area:
- **Small Objects:** $\text{Area} < 32^2 = 1,024\text{ px}^2$
- **Medium Objects:** $1,024\text{ px}^2 \le \text{Area} \le 96^2 = 9,216\text{ px}^2$
- **Large Objects:** $\text{Area} > 96^2 = 9,216\text{ px}^2$

Detecting small objects requires high-resolution feature pyramids (FPN) and fine-grained feature maps.


In [ ]:
scale_dist = (
    bbox_df.group_by("scale_category")
    .len()
    .rename({"len": "count"})
    .with_columns(((pl.col("count") / bbox_df.height) * 100).round(2).alias("percentage"))
    .sort("count", descending=True)
)
print("Scale category distribution:")
print(scale_dist)

## 5. Bounding Box Aspect Ratio Spectrum & Anchor Design

Object detection anchors must closely cover the empirical spectrum of target aspect ratios ($W/H$). For example, pedestrians typically exhibit $W/H \approx 0.4 - 0.5$ (tall and narrow), whereas vehicles exhibit $W/H \approx 1.2 - 2.2$ (wide and low).


In [ ]:
aspect_stats = (
    bbox_df.group_by("class_name")
    .agg(
        [
            pl.col("aspect_ratio").mean().round(3).alias("mean_ar"),
            pl.col("aspect_ratio").min().round(3).alias("min_ar"),
            pl.col("aspect_ratio").max().round(3).alias("max_ar"),
            pl.col("area").mean().round(1).alias("mean_area"),
            pl.len().alias("count"),
        ]
    )
    .sort("count", descending=True)
)
print("Per-class aspect ratio and area metrics:")
print(aspect_stats)

## 6. Visualizing Bounding Box Distributions

We construct an analytical visualization suite:
1. Aspect ratio distribution histogram.
2. Bounding box width vs. height scatter plot.
3. Object scale category breakdown.
4. Class frequency distribution.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Aspect Ratio Histogram
ar_values = bbox_df["aspect_ratio"].to_numpy()
axes[0, 0].hist(ar_values, bins=15, color="#1f77b4", edgecolor="black", alpha=0.7)
axes[0, 0].axvline(1.0, color="red", linestyle="--", label="Square (1:1)")
axes[0, 0].set_title("Bounding Box Aspect Ratio Distribution (W / H)")
axes[0, 0].set_xlabel("Aspect Ratio")
axes[0, 0].set_ylabel("Count")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Width vs Height Scatter Plot
w_vals = bbox_df["box_width"].to_numpy()
h_vals = bbox_df["box_height"].to_numpy()
axes[0, 1].scatter(w_vals, h_vals, color="#ff7f0e", alpha=0.6, edgecolors="none")
axes[0, 1].plot([0, 250], [0, 250], color="black", linestyle="--", label="W = H")
axes[0, 1].set_title("Box Width vs. Height Dimensions")
axes[0, 1].set_xlabel("Width (px)")
axes[0, 1].set_ylabel("Height (px)")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Scale Category Breakdown
scale_names = scale_dist["scale_category"].to_list()
scale_counts = scale_dist["count"].to_list()
colors = ["#2ca02c", "#d62728", "#9467bd"]
axes[1, 0].bar(scale_names, scale_counts, color=colors, edgecolor="black", alpha=0.8)
axes[1, 0].set_title("COCO Scale Tiers (Small / Medium / Large)")
axes[1, 0].set_ylabel("Annotation Count")
axes[1, 0].grid(axis="y", alpha=0.3)

# 4. Class Frequency Distribution (Long-Tail)
class_dist = (
    bbox_df.group_by("class_name").len().rename({"len": "count"}).sort("count", descending=True)
)
cls_names = class_dist["class_name"].to_list()
cls_counts = class_dist["count"].to_list()
axes[1, 1].barh(cls_names[::-1], cls_counts[::-1], color="#17becf", edgecolor="black", alpha=0.8)
axes[1, 1].set_title("Class Frequency (Long-Tail Imbalance)")
axes[1, 1].set_xlabel("Annotation Count")
axes[1, 1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visual Object Tracking (VOT) Trajectory Analysis

Visual Object Tracking benchmarks require analyzing temporal bounding box continuity across video frames. We examine trajectory smoothness, bounding box area fluctuations, and occlusion events.


In [ ]:
track_df = pl.read_csv(TRACK_PATH)
print("Tracking dataset overview:")
print(f"Total video frames: {track_df.height}")
print(track_df.head(5))

In [ ]:
# Filter tracking sequence for pedestrian crossing
seq_df = track_df.filter(pl.col("sequence_id") == "seq_pedestrian_crossing")

frames = seq_df["frame_index"].to_numpy()
areas = seq_df["area"].to_numpy()
confs = seq_df["confidence"].to_numpy()
occlusions = seq_df["occlusion"].to_numpy()

fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(frames, areas, color="#1f77b4", marker="o", markersize=4, label="Bounding Box Area (px^2)")
ax1.set_xlabel("Video Frame Index")
ax1.set_ylabel("Bounding Box Area", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")

# Highlight occlusion window
for idx, occ in enumerate(occlusions):
    if occ == 1:
        ax1.axvspan(frames[idx] - 0.5, frames[idx] + 0.5, color="red", alpha=0.15)

ax2 = ax1.twinx()
ax2.plot(
    frames,
    confs,
    color="#d62728",
    linestyle="--",
    marker="s",
    markersize=3,
    label="Tracker Confidence",
)
ax2.set_ylabel("Tracker Confidence", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")
ax2.set_ylim(0.4, 1.05)

plt.title("Visual Object Tracking: Scale Evolution and Occlusion Dynamics")
fig.tight_layout()
plt.show()

## 8. Temporal Subsampling and Stride Downsampling

Video datasets typically record at 30 to 60 FPS, containing excessive redundant frames. Temporal downsampling with a uniform stride $S$ reduces redundant computation while preserving motion dynamics.


In [ ]:
def subsample_sequence(df: pl.DataFrame, stride: int) -> pl.DataFrame:
    # Downsample a tracking sequence by taking every S-th frame.
    return df.filter(pl.col("frame_index") % stride == 0)


sampled_seq = subsample_sequence(seq_df, stride=3)
print(f"Original frames: {seq_df.height} -> Subsampled frames (stride=3): {sampled_seq.height}")
print("Subsampled frames list:", sampled_seq["frame_index"].to_list())

## 9. Data Labeling Ecosystem Overview

Preparing high-quality datasets for Computer Vision involves specialized annotation tools:

| Tool | Format & Domain | Key Strengths | Export Capabilities |
| :--- | :--- | :--- | :--- |
| **Universal Data Tool** | Multi-Modal (Image, Text, Audio) | Web/Desktop client, zero-setup, configurable UI | JSON, CSV, image classification manifests |
| **CVAT (Computer Vision Annotation Tool)** | Video & High-Density Image Annotations | Frame-by-frame interpolation, AI tracking models, multi-user server | YOLO, COCO JSON, Pascal VOC XML, TFRecord |
| **Label Studio** | Extensible Multi-Domain Platform | Active learning, machine learning backend integration, custom XML schemas | COCO, YOLO, CSV, JSON-Min |
| **PPOCRLabel** | OCR & Document Text | Rotated bounding boxes, polygon vertices, transcription validation | PaddleOCR format, text coordinates |


## 10. Practice Challenge: Automated Dataset Diagnostics

**Exercise:** Write a function `compute_dataset_diagnostics(df: pl.DataFrame) -> dict` that computes:
1. `total_annotations`: Total row count.
2. `unique_images`: Total distinct images.
3. `mean_density`: Average bounding boxes per image.
4. `small_pct`: Percentage of small objects (area $< 1024$).
5. `imbalance_ratio`: Ratio of the most frequent class count to the least frequent class count.


In [ ]:
def compute_dataset_diagnostics(df: pl.DataFrame) -> dict:
    total_annotations = df.height
    unique_images = df["image_id"].n_unique()
    mean_density = round(total_annotations / unique_images, 2)

    small_count = df.filter(pl.col("area") < 1024).height
    small_pct = round((small_count / total_annotations) * 100, 2)

    class_counts = df.group_by("class_name").len()["len"].to_list()
    imbalance_ratio = round(max(class_counts) / min(class_counts), 2)

    return {
        "total_annotations": total_annotations,
        "unique_images": unique_images,
        "mean_density": mean_density,
        "small_pct": small_pct,
        "imbalance_ratio": imbalance_ratio,
    }


# Execute and verify diagnostics
report = compute_dataset_diagnostics(bbox_df)
print("Automated Dataset Diagnostic Report:")
for k, v in report.items():
    print(f"  {k}: {v}")

assert report["total_annotations"] == bbox_df.height
assert report["unique_images"] == bbox_df["image_id"].n_unique()
print("All assertions passed!")